# Part 4 — Vector Databases: Embeddings & Semantic Similarity

**Topics covered:** Cricket, Cooking, Cybersecurity  
**Model:** `all-MiniLM-L6-v2` via `sentence-transformers`  
**Tasks:**
1. Generate embeddings for 10 sentences across 3 topics
2. Compute a 10×10 cosine similarity matrix and display as a heatmap
3. Find the top 2 most similar sentences to a new cricket query

In [ ]:
# Install required libraries
!pip install sentence-transformers -q
!pip install seaborn -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded successfully.')

## Step 1 — Define 10 Sentences Across 3 Topics

In [ ]:
# 10 sentences: 4 Cricket, 3 Cooking, 3 Cybersecurity
sentences = [
    # Cricket (indices 0-3)
    "The batsman hit a magnificent century in the final over.",
    "India won the Test match by a margin of seven wickets.",
    "The spinner delivered a sharp googly that confused the batsman.",
    "A perfect yorker at the death overs sealed the victory for the team.",

    # Cooking (indices 4-6)
    "Sauté the onions in olive oil until they turn golden brown.",
    "Marinate the chicken in yogurt and spices for at least two hours.",
    "Fold the egg whites gently into the batter to keep it light and fluffy.",

    # Cybersecurity (indices 7-9)
    "The hacker exploited a SQL injection vulnerability to access the database.",
    "Always enable two-factor authentication to protect your online accounts.",
    "Ransomware encrypted all company files and demanded a Bitcoin payment.",
]

# Labels for the heatmap
labels = [
    "Cricket-1", "Cricket-2", "Cricket-3", "Cricket-4",
    "Cooking-1", "Cooking-2", "Cooking-3",
    "Cyber-1",   "Cyber-2",   "Cyber-3",
]

print(f'Total sentences: {len(sentences)}')
for i, s in enumerate(sentences):
    print(f'  [{i}] {s}')

## Step 2 — Load Model and Generate Embeddings

In [ ]:
# Load the sentence-transformers model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded: all-MiniLM-L6-v2')

# Generate embeddings — each sentence becomes a 384-dimensional vector
embeddings = model.encode(sentences, show_progress_bar=True)

print(f'\nEmbedding shape: {embeddings.shape}')  # Expected: (10, 384)
print(f'Each sentence is represented as a {embeddings.shape[1]}-dimensional vector.')

## Step 3 — Compute 10×10 Cosine Similarity Matrix

In [ ]:
# Compute pairwise cosine similarity
similarity_matrix = cosine_similarity(embeddings)

print('Cosine Similarity Matrix (10x10):')
print(np.round(similarity_matrix, 3))

## Step 4 — Visualize as Heatmap

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    similarity_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='white',
    square=True,
    cbar_kws={'label': 'Cosine Similarity (0 = dissimilar, 1 = identical)'}
)

plt.title(
    '10×10 Cosine Similarity Matrix\nSentences from: Cricket | Cooking | Cybersecurity',
    fontsize=14, fontweight='bold', pad=15
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Heatmap saved as similarity_heatmap.png')
print('\nObservation: High similarity values (warm colours) cluster within')
print('the same topic blocks, confirming semantic grouping by embedding.')

## Step 5 — Query: Find Top 2 Most Similar Sentences

In [ ]:
# Query sentence (cricket-related, as specified in the assignment)
query = "The bowler took three wickets in one over."

# Embed the query
query_embedding = model.encode([query])

# Compute cosine similarity between query and all 10 sentences
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

print(f'Query: "{query}"')
print(f'\nSimilarity scores against all 10 sentences:')
for i, (sent, score) in enumerate(zip(sentences, query_similarities)):
    print(f'  [{labels[i]:12s}] Score: {score:.4f} | {sent}')

# Get top 2 most similar
top2_indices = np.argsort(query_similarities)[::-1][:2]

print(f'\n{"="*60}')
print(f'TOP 2 MOST SIMILAR SENTENCES TO THE QUERY:')
print(f'{"="*60}')
for rank, idx in enumerate(top2_indices, 1):
    print(f'\nRank {rank}: [{labels[idx]}]')
    print(f'  Sentence  : {sentences[idx]}')
    print(f'  Similarity: {query_similarities[idx]:.4f}')

## Summary

- **Model used:** `sentence-transformers/all-MiniLM-L6-v2` — a lightweight but powerful model that produces 384-dimensional embeddings optimised for semantic similarity tasks.
- **Observation:** The cosine similarity heatmap shows clearly visible high-similarity clusters within each topic (Cricket, Cooking, Cybersecurity) and low similarity between topics — confirming that the model captures semantic meaning, not just keyword overlap.
- **Query result:** The query `"The bowler took three wickets in one over"` correctly retrieves the top 2 most similar sentences from the Cricket topic, even though none of them share the exact words "bowler", "wickets", or "one over" — demonstrating semantic search in action.